# Simple RAG system 

In [1]:
!pip install -q -U langchain langchain-core langchain-community
!pip install -q -U langchain-huggingface langchain-chroma
!pip install -q -U transformers accelerate bitsandbytes 
!pip install -q -U sentence-transformers chromadb pypdf
!pip install -q -U langchain-text-splitters

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from transformers import BitsAndBytesConfig
import torch
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_chroma import Chroma 
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnablePassthrough
import textwrap

/tmp/ipykernel_22204/1044262226.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


## Model Name

In [3]:
name = "NousResearch/Llama-2-7b-chat-hf"

## Quantizing the Model to decrease the size of the model weights

In [4]:
compute_dtype = torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

## Initialising the tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    name,
    quantization_config=quantization_config,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


## Downloading the model

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    name,
    cache_dir="./model/",
    device_map="auto",
    dtype=compute_dtype,
    quantization_config=quantization_config,
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## Creating the pipeline 

In [7]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.4,
    top_p=0.95,
    repetition_penalty=1.15,
    do_sample=True,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'top_p', 'temperature', 'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [8]:
llm = HuggingFacePipeline(pipeline=pipe)

In [9]:
print(llm.invoke("What is the equation of radius of a Black Hole?"))

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



 nobody knows. The equation for the radius of a black hole is not known, as it depends on the mass and spin of the black hole in a way that is not yet fully understood by physicists. However, scientists have developed several mathematical models to describe the behavior of black holes based on their mass and spin, but these models do not provide an exact formula for the radius of a black hole.

The size of a black hole is determined by its event horizon, which marks the boundary beyond which nothing can escape the gravitational pull of the black hole. However, the event horizon does not correspond directly to a specific distance from the center of the black hole, as it is a two-dimensional surface rather than a one-dimensional circle. Therefore, there is no simple equation that can be used to calculate the radius of a black hole.

Instead, scientists use numerical simulations and observations of black holes to infer their properties, including their radii. These methods involve solvin

In [10]:
context_prompt = PromptTemplate.from_template(
    """ You are a question answering assistant. Answer the question based on the provided context.
        You are given the following context: {context}. Use this context to answer the question: {question}, Answer:""" 
)

## Documents

In [11]:
!mkdir -p ./data

In [12]:
!wget -c https://krishi.maharashtra.gov.in/Site/Upload/Pdf/MahaAgri%20AI%20Policy%202025%20-%202029%20Marathi.pdf -P data/

--2026-09-30 05:13:27--  https://krishi.maharashtra.gov.in/Site/Upload/Pdf/MahaAgri%20AI%20Policy%202025%20-%202029%20Marathi.pdf
Resolving krishi.maharashtra.gov.in (krishi.maharashtra.gov.in)... 20.204.21.162
Connecting to krishi.maharashtra.gov.in (krishi.maharashtra.gov.in)|20.204.21.162|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.



In [13]:
!wget -c https://krishi.maharashtra.gov.in/Site/Upload/Pdf/MahaAgri%20AI%20Policy%202025%20-%202029%20English.pdf -P data/

--2026-09-30 05:13:29--  https://krishi.maharashtra.gov.in/Site/Upload/Pdf/MahaAgri%20AI%20Policy%202025%20-%202029%20English.pdf
Resolving krishi.maharashtra.gov.in (krishi.maharashtra.gov.in)... 20.204.21.162
Connecting to krishi.maharashtra.gov.in (krishi.maharashtra.gov.in)|20.204.21.162|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.



In [14]:
!wget -c https://krishi.maharashtra.gov.in/Site/Upload/Pdf/Problem%20Statement.pdf -P data/

--2026-09-30 05:13:30--  https://krishi.maharashtra.gov.in/Site/Upload/Pdf/Problem%20Statement.pdf
Resolving krishi.maharashtra.gov.in (krishi.maharashtra.gov.in)... 20.204.21.162
Connecting to krishi.maharashtra.gov.in (krishi.maharashtra.gov.in)|20.204.21.162|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.



In [15]:
!wget -c https://krishi.maharashtra.gov.in/Site/Upload/Pdf/CFPBackground.pdf -P data/

--2026-09-30 05:13:31--  https://krishi.maharashtra.gov.in/Site/Upload/Pdf/CFPBackground.pdf
Resolving krishi.maharashtra.gov.in (krishi.maharashtra.gov.in)... 20.204.21.162
Connecting to krishi.maharashtra.gov.in (krishi.maharashtra.gov.in)|20.204.21.162|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.



In [16]:
loader = PyPDFDirectoryLoader("./data")


In [17]:
docs = loader.load()

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=10,
)
chunks = text_splitter.split_documents(docs)
print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 2480


In [19]:
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"

In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="agriwelfare",
    persist_directory="db"
)

In [22]:
retriever = vectordb.as_retriever(search_kwargs={"k": 4})

retrieved_docs = retriever.invoke(
    "List the problem statements called in Maha Agri AI Policy"
)

print("\n\n".join(
    f"[Document {i+1}]\n{doc.page_content}" for i, doc in enumerate(retrieved_docs)
))

[Document 1]
5
Identified Problem Statements for call for proposal un der Maha Agri AI Policy

[Document 2]
5
Identified Problem Statements for call for proposal un der Maha Agri AI Policy

[Document 3]
3
Identified Problem Statements for call for proposal un der Maha Agri AI Policy

[Document 4]
3
Identified Problem Statements for call for proposal un der Maha Agri AI Policy


In [23]:
def format_docs(docs):
    return "\n\n".join(
        f"[Document {i+1}]\n{doc.page_content}" for i, doc in enumerate(docs)
    )

retriever = vectordb.as_retriever(search_kwargs={"k": 4})
chain = (
    {
        "context" : retriever | RunnableLambda(format_docs),
            "question" : RunnablePassthrough(),
    }
    | context_prompt
    | llm
    | StrOutputParser()
)

In [24]:
def output(query):
    result = chain.invoke(query)
    return result.strip()

In [25]:
print(output("List the problem statements called in Maha Agri AI Policy"))

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on the provided context, the following problem statements were identified and called in Maha Agri AI Policy:

* Document 1: "Maha Agri AI Policy"
* Document 2: "Maha Agri AI Policy"
* Document 3: "Maha Agri AI Policy"
* Document 4: "Maha Agri AI Policy"


# GRADIO

In [26]:
!pip install -q gradio

In [27]:
import gradio as gr

In [28]:
with gr.Blocks() as demo:
    gr.Markdown(
        """# AGRIWELFARE """
    )
    with gr.Row():
        with gr.Column():
            query = gr.Textbox(label="Enter your query", placeholder="Ask a question about Maha Agri AI Policy")
            submit = gr.Button("Submit")
        with gr.Column():
            result = gr.Textbox(label="Answer", placeholder="The answer will be displayed here", interactive=False)
    with gr.Row():
        submit.click(fn=output, inputs=[query], outputs=result)

In [29]:
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ad4574d350b9a935a2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ad4574d350b9a935a2.gradio.live
